# ⚡ Gate-Kernel Fusion Pilot v2 — tl.dot (tensor-core) rewrite

Spec: `docs/GATE_KERNEL_FUSION_SPEC.md`. v1 (hand-rolled broadcast-multiply-
reduce) lost to the existing unfused path by ~7x (Section 8, real Kaggle
run, 2026-09-17) -- root cause: no tensor cores, and cuBLAS's GEMV was
already close to the memory-bandwidth floor, so there wasn't much slack to
fuse away, and the naive kernel was simply a worse GEMV on top of that.

v2 (`_fused_gated_lm_head_kernel_dot` in `netelpro/neuro/gate_kernel.py`)
uses `tl.dot` with the standard M=1→16 zero-padding trick to hit tensor
cores on a T4 (fp16 inputs, fp32 accumulator). This notebook benchmarks
**both kernels side by side** against the same baseline/ungated numbers,
so the comparison isolates the kernel-design change, not measurement noise.
No speedup claimed before it's measured here.

## 1. Hardware check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
assert torch.cuda.is_available(), "Este pilot necesita GPU real -- activa el acelerador GPU T4/P100 en Kaggle."

try:
    import triton
    print(f"Triton: {triton.__version__}")
except ImportError as e:
    raise RuntimeError(
        "Triton no está instalado en esta imagen de Kaggle. "
        "Intenta `!pip install -q triton` en la celda siguiente si esto falla."
    ) from e


## 2. Clonar/actualizar el repo e instalar

Repo privado -- necesita un GitHub token. En Kaggle: **Add-ons -> Secrets -> Add a new secret**, Label `GITHUB_TOKEN` (el nombre lo elegis vos, no es un nombre reservado; solo tiene que matchear el `get_secret(...)` de abajo), Value tu Personal Access Token (scope `repo`, solo lectura del codigo alcanza). Despues activa el secret para este notebook (toggle ON) antes de correr la celda.

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")

%cd /kaggle/working
import os
repo_url = f"https://{github_token}@github.com/jona2428/netelpro.git"
if not os.path.exists('netelpro'):
    !git clone {repo_url}
else:
    !cd /kaggle/working/netelpro && git remote set-url origin {repo_url} && git pull origin master
%cd /kaggle/working/netelpro
!pip install -q -e .


## 3. Construir Teo v2 en GPU

Config real de `docs/ROADMAP_TEO_7B_MOE_SDS.md` (vocab 32768, 12 capas, 12
cabezas, d_model 768). Init aleatorio -- este pilot mide latencia del kernel,
no calidad del modelo, así que no hace falta descargar ningún checkpoint.

In [ ]:
import torch
from netelpro.neuro.transformer import NetelproTransformer, NetelproTransformerConfig
from netelpro.neuro.gate_kernel import HAS_TRITON, gated_lm_head, gated_lm_head_naive, gated_lm_head_reference
from netelpro.neuro.native_kernel import NetelproVectorKernel
from pathlib import Path

assert HAS_TRITON, "HAS_TRITON es False -- revisa la instalación de triton en esta imagen."

config = NetelproTransformerConfig(
    vocab_size=32768, block_size=512, n_layer=12, n_head=12, n_embd=768,
)
model = NetelproTransformer(config).cuda().eval()
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parámetros")

vk = NetelproVectorKernel(Path("netelpro/neuro/rules/action_boundary.sl"))


## 4. Chequeo de correctitud en GPU real (antes de confiar en cualquier número)

Compara los dos kernels Triton (v1 naive, v2 `tl.dot`) contra la referencia
torch (misma regla, tres implementaciones) sobre tensores CUDA reales. v1
compara exacto (fp32 en todo el pipeline). v2 usa fp16 para los operandos
del `tl.dot` (así entra a los tensor cores de la T4) con acumulación fp32
-- tolerancia más laxa a propósito, y el máximo error absoluto/relativo se
imprime explícitamente en vez de asumir que "más o menos" alcanza.

In [ ]:
torch.manual_seed(0)
x = torch.randn(config.n_embd, device="cuda")
weight = model.lm_head.weight.detach()

test_ranges = [(0, config.vocab_size - 1, 1), (100, 30000, 1), (5000, 5200, 1), (0, config.vocab_size - 1, 0)]

all_ok = True
for lo, hi, safety in test_ranges:
    ref = gated_lm_head_reference(x, weight, lo, hi, safety)

    naive = gated_lm_head_naive(x, weight, lo, hi, safety)
    naive_ok = torch.allclose(naive, ref, atol=1e-3, rtol=1e-3) if safety == 1 else torch.equal(naive, ref)

    dot = gated_lm_head(x, weight, lo, hi, safety)
    if safety == 1:
        finite = torch.isfinite(ref)
        abs_diff = (dot[finite] - ref[finite]).abs()
        rel_diff = abs_diff / ref[finite].abs().clamp_min(1e-3)
        dot_ok = bool((abs_diff.max() < 0.5) and (rel_diff.max() < 0.05)) if finite.any() else True
        masked_ok = bool((dot[~finite] == float("-inf")).all())
        dot_ok = dot_ok and masked_ok
        print(f"range=({lo},{hi}) safety={safety}: naive_match={naive_ok}, "
              f"dot_match={dot_ok} (max_abs_diff={abs_diff.max().item() if finite.any() else 0:.4f}, "
              f"max_rel_diff={rel_diff.max().item() if finite.any() else 0:.4f})")
    else:
        dot_ok = torch.equal(dot, ref)
        print(f"range=({lo},{hi}) safety={safety}: naive_match={naive_ok}, dot_match={dot_ok}")

    all_ok &= naive_ok and dot_ok

assert all_ok, "Alguno de los kernels no coincide con la referencia -- NO confíes en los tiempos de la sección 5 hasta arreglar esto."
print("\nCorrectitud confirmada en GPU real (v1 exacto, v2 dentro de tolerancia fp16). Procediendo al benchmark.")


## 5. Benchmark de cuatro vías (números reales, no prometidos)

Para cada config de rango: `ungated` (matmul solo, el piso), `baseline`
(matmul + `filter_logits_tensor`, el path real de hoy), `fused_triton_naive`
(v1, ya sabemos que pierde -- referencia de comparación), `fused_triton_dot`
(v2, `tl.dot`/tensor cores -- lo nuevo). `torch.cuda.Event` para timing real
de GPU, promedio sobre varios pasos con warmup.

In [ ]:
def bench_step(fn, n_warmup=20, n_iters=200):
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    times_ms = []
    for _ in range(n_iters):
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        times_ms.append(start.elapsed_time(end))
    times_ms.sort()
    return {
        "mean_ms": sum(times_ms) / len(times_ms),
        "min_ms": times_ms[0],
        "max_ms": times_ms[-1],
        "p50_ms": times_ms[len(times_ms) // 2],
    }

x_last = torch.randn(config.n_embd, device="cuda")
weight = model.lm_head.weight.detach()

range_configs = {
    "broad (allowed [100, 30000] of 32768 -- mostly allowed)": (100, 30000, 1),
    "narrow (allowed [5000, 5200] of 32768 -- mostly masked)": (5000, 5200, 1),
}

results = {}
for label, (lo, hi, safety) in range_configs.items():
    results[label] = {
        "ungated": bench_step(lambda: torch.nn.functional.linear(x_last, weight)),
        "baseline_unfused": bench_step(lambda: vk.filter_logits_tensor(
            torch.nn.functional.linear(x_last, weight).unsqueeze(0), lo, hi, safety
        )),
        "fused_triton_naive": bench_step(lambda: gated_lm_head_naive(x_last, weight, lo, hi, safety)),
        "fused_triton_dot": bench_step(lambda: gated_lm_head(x_last, weight, lo, hi, safety)),
    }

import json
print(json.dumps(results, indent=2))


## 6. Tabla de resultados

In [ ]:
import pandas as pd

rows = []
for range_label, methods in results.items():
    for method, m in methods.items():
        rows.append({"range": range_label, "method": method, **m})

df = pd.DataFrame(rows)
df


## 7. Lectura honesta

Completar después de correr:
- ¿`fused_triton_dot` le ganó a `baseline_unfused`? ¿Y a `fused_triton_naive` (v1)?
- ¿Cuánto mejoró el diseño `tl.dot` sobre el naive, en cada config de rango?
- ¿Qué tan cerca quedó `fused_triton_dot` de `ungated` (el piso teórico)?
- Si esto corrió en P100 en vez de T4: ¿tl.dot igual ayudó sin tensor cores reales, o quedó igual que v1?

Pegar la tabla real de la celda 6 en `docs/GATE_KERNEL_FUSION_SPEC.md` Sección 9 cuando esté corrido -- no antes.